# 제7-2장. 이해관계자와 네트워크 분석

## 학습 목표
1. 이해관계자 분석의 필요성과 Freeman의 이해관계자 이론 이해
2. 영향력-관심도 매트릭스(Power-Interest Matrix)의 원리와 한계 파악
3. 사회 연결망 분석(SNA)의 핵심 개념과 중심성 지표 이해
4. NetworkX를 활용한 이해관계자 네트워크 분석 실습
5. 네트워크 기반 변화 관리 전략 수립 역량 함양

## 강의 구조
| 시간 | 내용 | 활동 |
|------|------|------|
| Part 1 | 이해관계자 분석 기초 | 이론 + 조사 과제 |
| Part 2 | 네트워크 분석 기초 | 이론 + 조사 과제 + 간단 실습 |
| ☕ 휴식 (15분) | | |
| Part 3 | 기획에서의 활용과 변화 관리 | 이론 + 조사 과제 |
| Part 4 | 실습: 이해관계자 매핑 | 코드 작성 |
| Part 5 | 실습: 네트워크 분석 | 코드 작성 |

In [ ]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly", "networkx"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import networkx as nx

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

## Part 1: 이해관계자 분석의 필요성

### Freeman의 이해관계자 이론 (1984)

> **이해관계자(Stakeholder)란** "조직의 목표 달성에 영향을 미치거나 영향을 받는 개인 또는 집단"을 의미한다.

이 정의는 단순한 분류를 넘어 **상호작용의 관점**을 제시한다:
- 조직이 이해관계자에게 영향을 미침 (기업의 정책, 제품, 활동)
- 이해관계자가 조직에 영향을 미침 (지지, 반발, 저항, 참여)

### 왜 이해관계자 분석이 필요한가?

#### 1. **실행 가능성 평가**
- 아무리 완벽한 기획도 이해관계자의 지지 없이는 실행되지 않음
- 반대 세력의 저항이 얼마나 강한지 미리 파악
- 필요한 동의와 지원을 예측

#### 2. **참여 전략 수립**
- 누가 핵심 의사결정자인가?
- 누가 정보를 많이 가지고 있는가?
- 누가 조용한 영향력을 행사하는가?

#### 3. **리스크 관리**
- 어느 이해관계자의 반발이 가장 위험한가?
- 어떤 연합이 형성될 수 있는가?
- 언제 저항이 임계점에 도달할 것인가?

### 이해관계자의 유형

#### 직접 이해관계자 (Direct Stakeholders)
계약, 소유, 고용 등으로 직접 연결
- 경영진, 직원, 주주, 고객, 공급업체

#### 간접 이해관계자 (Indirect Stakeholders)
정책, 규제, 사회적 영향으로 간접 연결
- 정부 부서, 규제 기관, 지역사회

#### 숨은 이해관계자 (Hidden Stakeholders)
비공식 네트워크로 영향을 행사
- 현 경영진의 고문, 언론 기자, 활동가, 인플루언서

### 전통적 접근법의 한계

단순히 이해관계자를 목록화하는 것은 충분하지 않다:
- **정적(Static)**: 한 시점의 스냅샷, 시간 경과에 따른 변화 미반영
- **태도 미반영**: Power-Interest만으로는 '무엇을 해야 할까?'를 답하기 어려움
- **관계 무시**: 개별 이해관계자를 독립적으로 취급, 네트워크 구조 간과
- **영향력 오류**: 공식적 권한만 중시, 비공식 영향력 간과

## Power-Interest Matrix (Mendelow, 1991)

### 4분면 분류 체계

| **4분면** | **특징** | **전략** |
|----------|---------|----------|
| **Key Players**<br>(높은 권력 + 높은 관심도) | 조직 내 영향력이 크고 이슈에 깊이 관여 | 적극적 참여, 핵심 이해관계자로 관리 |
| **Keep Satisfied**<br>(높은 권력 + 낮은 관심도) | 결정적 영향력 있지만 현재 관심 낮음 | 만족도 유지, 장기적 지지 확보 |
| **Keep Informed**<br>(낮은 권력 + 높은 관심도) | 영향력은 적지만 열정적 참여 의지 | 정보 투명 공개, 의견 청취 |
| **Minimal Effort**<br>(낮은 권력 + 낮은 관심도) | 현재 영향력과 관심도 모두 낮음 | 모니터링만 지속 |

### Power (권력) 평가 요소

1. **공식적 권한 (Formal Authority)**
   - 조직 내 직위, 의사결정 권한, 예산 통제력
   - 예: 시장, 국장, 임원진

2. **자원 통제력 (Resource Control)**
   - 자금, 인력, 기술, 정보 등의 통제
   - 예: 재무부서장, 기술보유 기업

3. **전문성 (Expertise)**
   - 특정 분야의 깊은 지식, 신뢰도
   - 예: 환경 전문가, 컨설턴트

4. **네트워크 (Network)**
   - 다른 영향력자와의 연결, 여론 주도력
   - 예: 언론, 인플루언서, 협회장

### 한계점 (Limitations)

이 매트릭스만으로는 불충분하다:

1. **태도(Attitude) 미반영**
   - 같은 Power-Interest라도 지지자와 반대자는 전혀 다른 전략 필요
   - "Keep Satisfied"도 반대 세력이면 매우 위험

2. **관계 구조 무시**
   - 이해관계자 간 연결과 영향이 드러나지 않음
   - "연결 없는 반대자 5명"과 "연결된 반대자 1명"의 리스크는 다름

3. **동적 변화 미처리**
   - 정보 공개, 신문 보도 등으로 관심도가 급격히 변할 수 있음
   - 네트워크 효과(cascade)로 지지/반대가 확산될 수 있음

4. **비공식 영향력 누락**
   - 공식 직위는 낮지만 실제 영향력이 큰 인물 간과
   - "실세" 또는 "보좌관"의 역할 미반영

In [ ]:
# 탄소중립 도시 정책 이해관계자 데이터
stakeholders = pd.DataFrame({
    'id': ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 
           'S09', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15'],
    'name_en': [
        'Mayor', 'Env. Minister', 'Env. Director', 'Finance Director', 'Planning Director',
        'Env. NGO A', 'Env. NGO B', 'Large Manufacturer', 'SME Association',
        'Local Media', 'Env. Expert', 'Univ. Research Team', 'Chamber of Commerce',
        'City Council', 'General Public'
    ],
    'name_kr': ['시장', '환경부 장관', '환경국장', '재정국장', '기획조정실장',
                '환경시민단체 A', '환경시민단체 B', '대형 제조업체', '중소기업협회',
                '지역 언론사', '환경 전문가', '대학 연구팀', '지역 상공회의소',
                '시의회 의원', '일반 시민'],
    'category': [
        'Government', 'Government', 'Government', 'Government', 'Government',
        'Civil Society', 'Civil Society', 'Business', 'Business',
        'Media', 'Expert', 'Expert', 'Business', 'Politics', 'Citizen'
    ],
    'power': [95, 85, 70, 75, 65, 40, 35, 80, 55, 60, 45, 40, 50, 70, 25],
    'interest': [80, 75, 90, 40, 50, 95, 90, 60, 45, 70, 85, 80, 55, 65, 35],
    'attitude': [1, 1, 1, -1, 0, 1, 1, -1, -1, 0, 1, 1, -1, 0, 0]  # 1:지지, 0:중립, -1:반대
})

# 색상 매핑 (태도)
color_map = {1: 'rgba(46, 204, 113, 0.8)', 0: 'rgba(149, 165, 166, 0.6)', -1: 'rgba(231, 76, 60, 0.8)'}
attitude_text = {1: 'Support', 0: 'Neutral', -1: 'Oppose'}

# 심볼 매핑 (카테고리)
symbol_map = {
    'Government': 'square', 'Civil Society': 'diamond', 'Business': 'triangle-up',
    'Media': 'cross', 'Expert': 'star', 'Politics': 'hexagon', 'Citizen': 'circle'
}

fig = go.Figure()

# 이해관계자 점 추가
for idx, row in stakeholders.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['power']], y=[row['interest']],
        mode='markers+text',
        marker=dict(
            size=15,
            color=color_map[row['attitude']],
            symbol=symbol_map[row['category']],
            line=dict(color='white', width=2)
        ),
        text=[row['name_en']], textposition='top center',
        textfont=dict(size=8),
        hovertemplate='<b>%{text}</b><br>' +
                      'Power: %{x}<br>' +
                      'Interest: %{y}<br>' +
                      'Attitude: ' + attitude_text[row['attitude']] +
                      '<extra></extra>',
        name=attitude_text[row['attitude']],
        showlegend=False
    ))

# 4분면 구분선 추가
fig.add_shape(type='line', x0=50, x1=50, y0=0, y1=100,
              line=dict(color='black', width=2, dash='dash'))
fig.add_shape(type='line', x0=0, x1=100, y0=50, y1=50,
              line=dict(color='black', width=2, dash='dash'))

# 4분면 라벨 추가
fig.add_annotation(text='Keep Satisfied', x=75, y=25, showarrow=False,
                   font=dict(size=12, color='rgba(0,0,0,0.3)'))
fig.add_annotation(text='Key Players', x=75, y=75, showarrow=False,
                   font=dict(size=12, color='rgba(0,0,0,0.3)'))
fig.add_annotation(text='Minimal Effort', x=25, y=25, showarrow=False,
                   font=dict(size=12, color='rgba(0,0,0,0.3)'))
fig.add_annotation(text='Keep Informed', x=25, y=75, showarrow=False,
                   font=dict(size=12, color='rgba(0,0,0,0.3)'))

# 범례 추가
for attitude, color in color_map.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color=color, symbol='circle'),
        name=attitude_text[attitude],
        showlegend=True
    ))

fig.update_layout(
    title='Power-Interest Matrix: Carbon-Neutral City Policy Stakeholders',
    xaxis=dict(title='Power', range=[0, 100]),
    yaxis=dict(title='Interest', range=[0, 100]),
    width=900, height=700,
    hovermode='closest',
    template='plotly_white',
    legend=dict(x=1.05, y=1)
)

fig.show()

print("\n💡 해석:")
print("  • 시장(S01), 환경국장(S03): Key Players - 적극적 참여와 지지 필요")
print("  • 환경시민단체(S06, S07): Keep Informed - 투명한 정보 공개로 신뢰 유지")
print("  • 재정국장(S04), 대기업(S08): 반대 세력 - 관심도가 낮지만 권력이 크다")
print("  • 기획조정실장(S05)은 가교 역할 (중간 위치, 중립)")

In [ ]:
# 태도별 분석
stakeholders['attitude_label'] = stakeholders['attitude'].map({1: 'Support', 0: 'Neutral', -1: 'Oppose'})
stakeholders['quadrant'] = stakeholders.apply(
    lambda row: 'Key Players' if (row['power'] >= 50 and row['interest'] >= 50)
                else 'Keep Satisfied' if (row['power'] >= 50 and row['interest'] < 50)
                else 'Keep Informed' if (row['power'] < 50 and row['interest'] >= 50)
                else 'Minimal Effort', axis=1
)

# 태도별로 정렬
order = {'Support': 0, 'Neutral': 1, 'Oppose': 2}
analysis = stakeholders.sort_values(by=['attitude', 'power'], ascending=[True, False])

fig = go.Figure()

for attitude in [-1, 0, 1]:
    subset = analysis[analysis['attitude'] == attitude].sort_values('power', ascending=True)
    label = attitude_text[attitude]
    color = color_map[attitude]
    fig.add_trace(go.Bar(
        y=subset['name_kr'],
        x=subset['power'],
        orientation='h',
        name=label,
        marker=dict(color=color),
        hovertemplate='%{y}<br>Power: %{x}<extra></extra>'
    ))

fig.update_layout(
    title='Risk Assessment: Power Level by Stakeholder Attitude',
    xaxis_title='Power Level',
    yaxis_title='Stakeholder',
    barmode='group',
    width=1000, height=600,
    template='plotly_white',
    hovermode='closest'
)

fig.show()

print("\n🚨 고위험 이해관계자 (높은 권력 + 반대):")
high_risk = stakeholders[(stakeholders['power'] >= 70) & (stakeholders['attitude'] == -1)]
for idx, row in high_risk.iterrows():
    print(f"  • {row['name_kr']} (Power: {row['power']}) - {row['category']}")

print("\n✅ 전략적 지지자 (높은 권력 + 지지):")
strong_support = stakeholders[(stakeholders['power'] >= 70) & (stakeholders['attitude'] == 1)]
for idx, row in strong_support.iterrows():
    print(f"  • {row['name_kr']} (Power: {row['power']}) - {row['category']}")

### 📝 이론 과제 7-2-1 (15분)

#### 과제: 이해관계자 분석의 기초 개념

**질문:**

1. **Freeman(1984)의 이해관계자 이론의 핵심 주장**
   - Freeman이 제시한 이해관계자의 정의가 무엇이며, 전통적 주주중심 경영과 어떤 점에서 다른가?
   - 조직이 이해관계자를 고려해야 하는 이유를 2-3문장으로 설명하시오.

2. **Power-Interest Matrix의 한계점**
   - Power-Interest Matrix만으로 이해관계자 분석이 부족한 이유 3가지를 찾아 각각 설명하시오.
   - 실제 기획 현장에서 이 한계가 어떤 문제를 야기할 수 있는가?

3. **숨은 이해관계자의 사례 조사**
   - 당신이 경험하거나 알고 있는 정책/프로젝트에서 "공식적 권한은 없지만 실제 영향력이 큰" 이해관계자의 사례를 찾아 설명하시오.
   - 그 사람의 영향력 원천이 무엇인지 분석하시오.

**조사 키워드:**
- "stakeholder theory Freeman"
- "Power-Interest Matrix limitations"
- "hidden stakeholders examples"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 7-2-1 제출란

**학번:** ___________
**이름:** ___________

#### 1. Freeman(1984)의 이해관계자 이론의 핵심 주장

(여기에 작성)

---

#### 2. Power-Interest Matrix의 한계점

(여기에 작성)

---

#### 3. 숨은 이해관계자의 사례

(여기에 작성)

## Part 2: 사회 연결망 분석 (Social Network Analysis, SNA) 기초

### 핵심 개념

#### **노드(Node)와 엣지(Edge)**
- **노드**: 이해관계자, 개인, 조직 등 네트워크의 구성 요소
- **엣지**: 노드 간의 관계 (communication, support, conflict, etc.)
- **무게(Weight)**: 관계의 강도 (자주 만나는가? 신뢰도는? 영향력은?)
- **방향성(Direction)**: 관계가 일방향인가 쌍방향인가?

### 네트워크 분석의 역사

| **시기** | **개발자/개념** | **의의** |
|---------|--------------|----------|
| 1930s | Moreno, Sociometry | 최초의 사회 연결망 분석 |
| 1960s-1970s | Granovetter, Strength of Weak Ties | 약한 연결의 힘 발견 |
| 1992 | Burt, Structural Holes | 구조적 공백과 정보 이점 |
| 2000s- | 온라인 네트워크, 빅데이터 | 실시간 분석, 대규모 데이터 |

### 4가지 중심성 지표 (Centrality Measures)

#### 1. **Degree Centrality (연결 중심성)**
$$C^D(v) = \frac{deg(v)}{n-1}$$

- **의미**: "얼마나 많은 사람과 직접 연결되어 있는가?"
- **해석**: 높을수록 사교적, 영향 범위가 넓음
- **기획 활용**: 정보 전파의 확산 속도, 영향력 범위

#### 2. **Betweenness Centrality (매개 중심성)**
$$C^B(v) = \sum_{s \neq v \neq t} \frac{\sigma_{st}(v)}{\sigma_{st}}$$

- **의미**: "다른 노드 간의 최단경로에 얼마나 자주 위치하는가?"
- **해석**: 높을수록 브로커 역할, 정보 게이트키퍼
- **기획 활용**: 연결고리 역할, 갈등 중보자, 정보 통제자

#### 3. **Closeness Centrality (근접 중심성)**
$$C^C(v) = \frac{n-1}{\sum_u d(v,u)}$$

- **의미**: "다른 노드들과 평균적으로 얼마나 가까운가?"
- **해석**: 높을수록 네트워크 중앙, 전체와의 거리 짧음
- **기획 활용**: 리더십 잠재력, 정책 전파 속도

#### 4. **Eigenvector Centrality (아이젠벡터 중심성)**
- **의미**: "중요한 노드와 연결되어 있는가?"
- **해석**: 낮은 연결도라도 강력한 노드와 연결되면 높음
- **기획 활용**: 영향력 있는 집단과의 관계, 권력 자산

### Structural Holes (구조적 공백)

**Burt(1992)의 이론:**

네트워크에서 **분리된 영역 간의 격차**를 관리하는 사람이 큰 이점을 가진다.

**3가지 이점:**
1. **정보 이점 (Information Benefit)**
   - 여러 영역의 정보를 먼저 접할 수 있음
   - 예: 정부, 기업, 시민단체를 연결하는 사람은 각 진영의 의도를 알 수 있음

2. **통제 이점 (Control Benefit)**
   - 메시지를 필터링하고 중개할 수 있음
   - 예: 기획자가 의견 조율의 중심이 되어 결과를 유리하게 도형

3. **시각 이점 (Vision Benefit)**
   - 여러 관점을 종합하여 새로운 아이디어 창출
   - 예: 다양한 이해관계자의 요구를 결합한 혁신안 제시

### Strength of Weak Ties (약한 연결의 힘)

**Granovetter(1973)의 발견:**

- **강한 연결(Strong Ties)**: 자주 만나고, 유사한 사람들
  - 장점: 신뢰, 깊은 정보
  - 단점: 같은 정보만 반복, 새로운 기회 제한

- **약한 연결(Weak Ties)**: 가끔 만나고, 다른 배경의 사람들
  - 장점: 새로운 정보, 다른 진영과의 신뢰 다리, 영향력 확대
  - 단점: 신뢰도 낮음, 깊이 부족

**기획에서의 함의:**
- 기획자는 약한 연결 파트너(다양한 분야의 전문가, 다른 부서 담당자)를 적극 활용해야 함
- 반목하는 진영 간 "약한 연결"을 만드는 것만으로도 갈등 완화 가능

### 커뮤니티 탐지 (Community Detection)

**Louvain Algorithm:**
- 네트워크를 **자동으로 군집(커뮤니티)**으로 분할
- **Modularity** 최대화: 군집 내 연결은 강하고, 군집 간 연결은 약하도록
- **기획 활용**: 파벌, 연합, 이익 집단 자동 식별

**전역 네트워크 특성:**
- **밀도(Density)**: 실제 연결 / 가능한 모든 연결
  - 높음 = 응집력 강한 네트워크 (합의하기 쉬움)
  - 낮음 = 분화된 네트워크 (독립적 행동 가능)
- **평균 경로 길이(Avg Path Length)**: 모든 노드 쌍 간 거리의 평균
  - 짧음 = 빠른 정보 전파, "작은 세상" 현상
- **클러스터링 계수(Clustering Coefficient)**: 삼각형 비율
  - 높음 = 강한 로컬 커뮤니티, 신뢰 기반

### 비교: 4가지 중심성 지표의 실무 활용

| **중심성** | **의미** | **기획에서의 활용** | **대상자** |
|----------|--------|------------------|----------|
| **Degree** | 직접 연결 수 | 정보 전파 범위 | 커뮤니케이션 허브 |
| **Betweenness** | 중개 역할 | 갈등 중보, 신뢰 구축 | 브로커, 중개자 |
| **Closeness** | 전체와의 거리 | 리더십 잠재력, 조율 능력 | 통합 리더 |
| **Eigenvector** | 중요 노드와의 연결 | 정치력, 권력 기반 | 영향력 엘리트 |

**실무 팁:**
- 조기 도입자? → **Degree + Eigenvector** 높은 사람
- 반대 세력 약화? → **Betweenness** 높은 브로커를 설득
- 전체 통합? → **Closeness** 높은 사람을 중심에
- 빠른 전파? → **Degree** 높은 사람을 먼저 설득

In [ ]:
# 이해관계자 네트워크 구축
def create_stakeholder_network():
    """탄소중립 도시 정책 이해관계자 네트워크 생성"""
    
    G = nx.Graph()
    
    # 노드 정보 (15개)
    nodes = {
        'S01': {'name_en': 'Mayor', 'name_kr': '시장', 'category': 'Government', 'attitude': 1},
        'S02': {'name_en': 'Env. Minister', 'name_kr': '환경부 장관', 'category': 'Government', 'attitude': 1},
        'S03': {'name_en': 'Env. Director', 'name_kr': '환경국장', 'category': 'Government', 'attitude': 1},
        'S04': {'name_en': 'Finance Director', 'name_kr': '재정국장', 'category': 'Government', 'attitude': -1},
        'S05': {'name_en': 'Planning Director', 'name_kr': '기획조정실장', 'category': 'Government', 'attitude': 0},
        'S06': {'name_en': 'Env. NGO A', 'name_kr': '환경시민단체 A', 'category': 'Civil Society', 'attitude': 1},
        'S07': {'name_en': 'Env. NGO B', 'name_kr': '환경시민단체 B', 'category': 'Civil Society', 'attitude': 1},
        'S08': {'name_en': 'Large Manufacturer', 'name_kr': '대형 제조업체', 'category': 'Business', 'attitude': -1},
        'S09': {'name_en': 'SME Association', 'name_kr': '중소기업협회', 'category': 'Business', 'attitude': -1},
        'S10': {'name_en': 'Local Media', 'name_kr': '지역 언론사', 'category': 'Media', 'attitude': 0},
        'S11': {'name_en': 'Env. Expert', 'name_kr': '환경 전문가', 'category': 'Expert', 'attitude': 1},
        'S12': {'name_en': 'Univ. Research Team', 'name_kr': '대학 연구팀', 'category': 'Expert', 'attitude': 1},
        'S13': {'name_en': 'Chamber of Commerce', 'name_kr': '지역 상공회의소', 'category': 'Business', 'attitude': -1},
        'S14': {'name_en': 'City Council', 'name_kr': '시의회 의원', 'category': 'Politics', 'attitude': 0},
        'S15': {'name_en': 'General Public', 'name_kr': '일반 시민', 'category': 'Citizen', 'attitude': 0}
    }
    
    # 노드 추가
    for node_id, attrs in nodes.items():
        G.add_node(node_id, **attrs)
    
    # 엣지 (29개, 가중치 1-5)
    edges = [
        # 정부 내부
        ('S01', 'S02', 4), ('S01', 'S03', 5), ('S01', 'S04', 4), ('S01', 'S05', 5),
        ('S01', 'S14', 4), ('S02', 'S03', 4), ('S03', 'S04', 3), ('S03', 'S05', 3),
        ('S04', 'S05', 4),
        # 시민사회
        ('S06', 'S07', 5), ('S06', 'S11', 4), ('S06', 'S12', 3),
        ('S07', 'S11', 3), ('S07', 'S15', 2),
        # 기업
        ('S08', 'S09', 4), ('S08', 'S13', 5), ('S09', 'S13', 4),
        # 크로스 연결
        ('S03', 'S06', 3), ('S03', 'S11', 4), ('S04', 'S08', 3),
        ('S04', 'S13', 3), ('S10', 'S01', 3), ('S10', 'S06', 2),
        ('S10', 'S08', 2), ('S11', 'S12', 5), ('S14', 'S04', 3),
        ('S14', 'S06', 2), ('S05', 'S14', 3), ('S12', 'S03', 3),
    ]
    
    for source, target, weight in edges:
        G.add_edge(source, target, weight=weight)
    
    return G

# 네트워크 생성
G = create_stakeholder_network()

print(f"✅ 네트워크 생성 완료")
print(f"  - 노드: {G.number_of_nodes()}개")
print(f"  - 엣지: {G.number_of_edges()}개")
print(f"  - 밀도: {nx.density(G):.3f}")

In [ ]:
# 중심성 지표 계산
degree_cent = nx.degree_centrality(G)
betweenness_cent = nx.betweenness_centrality(G, weight='weight')
closeness_cent = nx.closeness_centrality(G, distance='weight')
eigenvector_cent = nx.eigenvector_centrality(G, weight='weight', max_iter=500)

# 결과를 DataFrame으로 정리
centrality_data = []
for node_id in sorted(G.nodes()):
    node = G.nodes[node_id]
    centrality_data.append({
        'ID': node_id,
        'Name': node['name_kr'],
        'Category': node['category'],
        'Attitude': attitude_text[node['attitude']],
        'Degree': round(degree_cent[node_id], 3),
        'Betweenness': round(betweenness_cent[node_id], 3),
        'Closeness': round(closeness_cent[node_id], 3),
        'Eigenvector': round(eigenvector_cent[node_id], 3)
    })

centrality_df = pd.DataFrame(centrality_data)

# Plotly 테이블로 시각화
fig = go.Figure(data=[go.Table(
    header=dict(
        values=['ID', 'Name', 'Category', 'Attitude', 'Degree', 'Betweenness', 'Closeness', 'Eigenvector'],
        fill_color='paleturquoise',
        align='left',
        font=dict(size=11)
    ),
    cells=dict(
        values=[centrality_df[col] for col in centrality_df.columns],
        fill_color='lavender',
        align='left',
        font=dict(size=10)
    )
)])

fig.update_layout(
    title='Centrality Metrics for All Stakeholders',
    width=1200, height=600
)

fig.show()

print("\n💡 주요 발견:")
print(f"  • 최고 Degree: {centrality_df.loc[centrality_df['Degree'].idxmax(), 'Name']} ({centrality_df['Degree'].max():.3f})")
print(f"  • 최고 Betweenness: {centrality_df.loc[centrality_df['Betweenness'].idxmax(), 'Name']} ({centrality_df['Betweenness'].max():.3f})")
print(f"  • 최고 Closeness: {centrality_df.loc[centrality_df['Closeness'].idxmax(), 'Name']} ({centrality_df['Closeness'].max():.3f})")
print(f"  • 최고 Eigenvector: {centrality_df.loc[centrality_df['Eigenvector'].idxmax(), 'Name']} ({centrality_df['Eigenvector'].max():.3f})")

In [ ]:
# Spring layout으로 노드 좌표 계산
pos = nx.spring_layout(G, k=2, iterations=50, seed=42, weight='weight')

# 엣지 trace
edge_x = []
edge_y = []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.append(x0)
    edge_x.append(x1)
    edge_x.append(None)
    edge_y.append(y0)
    edge_y.append(y1)
    edge_y.append(None)

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    mode='lines',
    line=dict(width=1, color='rgba(100, 100, 100, 0.3)'),
    hoverinfo='none',
    showlegend=False
)

# 노드 trace
node_x = []
node_y = []
node_color = []
node_size = []
node_hover = []
node_symbol = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    
    # 색상: 태도
    node_color.append(color_map[G.nodes[node]['attitude']])
    
    # 크기: 아이젠벡터 중심성
    node_size.append(max(10, eigenvector_cent[node] * 50))
    
    # 호버 텍스트
    hover_text = (
        f"<b>{G.nodes[node]['name_kr']}</b><br>"
        f"Category: {G.nodes[node]['category']}<br>"
        f"Degree: {degree_cent[node]:.3f}<br>"
        f"Betweenness: {betweenness_cent[node]:.3f}<br>"
        f"Closeness: {closeness_cent[node]:.3f}<br>"
        f"Eigenvector: {eigenvector_cent[node]:.3f}"
    )
    node_hover.append(hover_text)
    
    # 심볼: 카테고리
    node_symbol.append(symbol_map[G.nodes[node]['category']])

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    marker=dict(
        size=node_size,
        color=node_color,
        symbol=node_symbol,
        line=dict(width=2, color='white')
    ),
    text=node_hover,
    hovertemplate='%{text}<extra></extra>',
    showlegend=False
)

# 라벨
label_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='text',
    text=[G.nodes[node]['name_en'] for node in G.nodes()],
    textposition='top center',
    textfont=dict(size=8),
    showlegend=False
)

# Figure 구성
fig = go.Figure(data=[edge_trace, node_trace, label_trace])

fig.update_layout(
    title='Stakeholder Network Visualization<br>(Node Size: Eigenvector Centrality, Color: Attitude)',
    showlegend=False,
    hovermode='closest',
    width=1000, height=900,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)

fig.show()

print("\n📊 네트워크 특성:")
print(f"  • 평균 clustering coefficient: {nx.average_clustering(G):.3f}")
print(f"  • 평균 경로 길이: {nx.average_shortest_path_length(G):.3f}")

In [ ]:
# 중심성 상위 5인 선택
top_nodes = centrality_df.nlargest(5, 'Eigenvector')['ID'].tolist()

# 정규화 (0-1 범위)
def normalize_centrality(df, columns):
    normalized = df.copy()
    for col in columns:
        max_val = df[col].max()
        if max_val > 0:
            normalized[col] = df[col] / max_val
    return normalized

norm_df = normalize_centrality(centrality_df, ['Degree', 'Betweenness', 'Closeness', 'Eigenvector'])

fig = go.Figure()

colors = ['rgb(46, 204, 113)', 'rgb(231, 76, 60)', 'rgb(52, 152, 219)', 'rgb(241, 196, 15)', 'rgb(155, 89, 182)']

for idx, node_id in enumerate(top_nodes):
    row = norm_df[norm_df['ID'] == node_id].iloc[0]
    fig.add_trace(go.Scatterpolar(
        r=[row['Degree'], row['Betweenness'], row['Closeness'], row['Eigenvector']],
        theta=['Degree', 'Betweenness', 'Closeness', 'Eigenvector'],
        fill='toself',
        name=row['Name'],
        line=dict(color=colors[idx])
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Centrality Profile Comparison (Top 5 Stakeholders)',
    width=800, height=600,
    showlegend=True
)

fig.show()

print("\n💡 중심성 프로필 해석:")
print("\n  • 환경국장(S03): Degree ↑ + Closeness ↑ = 허브 역할, 정보 전파 중심")
print("  • 환경시민단체 A(S06): Betweenness ↑ = 브로커, 진영 간 중개자")
print("  • 시장(S01): Eigenvector ↑ = 영향력 있는 네트워크 중심")
print("  • 재정국장(S04): 반대 세력 + 높은 권력 = 고위험 이해관계자")
print("  • 시의회 의원(S14): 정치와 정부를 연결하는 브로커")

### 📝 이론 과제 7-2-2 (15분)

#### 과제: 네트워크 분석 이론의 이해

**질문:**

1. **Burt(1992)의 구조적 공백 이론**
   - 구조적 공백의 3가지 이점(정보, 통제, 시각)을 각각 설명하시오.
   - 기획 현장에서 "구조적 공백을 활용"한다는 것이 구체적으로 무엇인지 예를 들어 설명하시오.

2. **Granovetter(1973)의 약한 연결의 힘**
   - "약한 연결"이 강한 연결보다 새로운 정보 전달에 효과적인 이유를 설명하시오.
   - Granovetter의 구직 연구 사례를 찾아 요약하시오. (당신이 현재 하는 일 또는 당신의 친구의 취직 사례)

3. **4가지 중심성 지표의 활용**
   - 각 중심성 지표를 기획 현장의 구체적인 전략 목표와 매칭하는 테이블을 작성하시오.
   - 예: Degree가 높은 사람은 "빠른 정보 전파"에 활용 → 조기 도입자로 선발

**조사 키워드:**
- "structural holes Burt benefits"
- "strength of weak ties Granovetter job search"
- "network centrality measures application"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 7-2-2 제출란

**학번:** ___________
**이름:** ___________

#### 1. Burt의 구조적 공백 이론과 기획에서의 활용

(여기에 작성)

---

#### 2. Granovetter의 약한 연결과 구직 사례

(여기에 작성)

---

#### 3. 중심성 지표의 전략적 활용 매칭표

(테이블 형식으로 작성)

In [ ]:
# Louvain 알고리즘으로 커뮤니티 탐지
communities = list(louvain_communities(G, seed=42))

# 커뮤니티 할당
community_map = {}
for comm_idx, comm in enumerate(communities):
    for node in comm:
        community_map[node] = comm_idx

print(f"🔍 커뮤니티 탐지 결과: {len(communities)}개 커뮤니티 발견\n")

for comm_idx, comm in enumerate(communities):
    print(f"커뮤니티 {comm_idx + 1}: {len(comm)}명")
    nodes_in_comm = [G.nodes[node]['name_kr'] for node in comm]
    print(f"  구성원: {', '.join(nodes_in_comm)}")
    
    # 태도 분석
    attitudes = [G.nodes[node]['attitude'] for node in comm]
    support = attitudes.count(1)
    neutral = attitudes.count(0)
    oppose = attitudes.count(-1)
    print(f"  태도: 지지 {support}명, 중립 {neutral}명, 반대 {oppose}명")
    print()

In [ ]:
# 커뮤니티별 색상
community_colors = ['rgb(46, 204, 113)', 'rgb(231, 76, 60)', 'rgb(52, 152, 219)', 'rgb(241, 196, 15)', 'rgb(155, 89, 182)']

# 엣지 trace
edge_x = []
edge_y = []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.append(x0)
    edge_x.append(x1)
    edge_x.append(None)
    edge_y.append(y0)
    edge_y.append(y1)
    edge_y.append(None)

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    mode='lines',
    line=dict(width=1, color='rgba(100, 100, 100, 0.2)'),
    hoverinfo='none',
    showlegend=False
)

# 노드 trace (커뮤니티별)
node_traces = []
for comm_idx, comm in enumerate(communities):
    node_x = [pos[node][0] for node in comm]
    node_y = [pos[node][1] for node in comm]
    node_hover = [f"<b>{G.nodes[node]['name_kr']}</b><br>Community {comm_idx + 1}" for node in comm]
    
    node_traces.append(go.Scatter(
        x=node_x, y=node_y,
        mode='markers',
        marker=dict(
            size=15,
            color=community_colors[comm_idx % len(community_colors)],
            line=dict(width=2, color='white')
        ),
        text=node_hover,
        hovertemplate='%{text}<extra></extra>',
        name=f'Community {comm_idx + 1}',
        showlegend=True
    ))

# 라벨
node_x = [pos[node][0] for node in G.nodes()]
node_y = [pos[node][1] for node in G.nodes()]
label_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='text',
    text=[G.nodes[node]['name_en'] for node in G.nodes()],
    textposition='top center',
    textfont=dict(size=7),
    showlegend=False
)

fig = go.Figure(data=[edge_trace] + node_traces + [label_trace])

fig.update_layout(
    title='Stakeholder Network by Community (Louvain Algorithm)',
    showlegend=True,
    hovermode='closest',
    width=1000, height=900,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)

fig.show()

## ☕ 휴식 (15분)

잠깐 쉬면서 Part 1, 2에서 배운 내용을 정리해 보세요.

### 여기까지의 핵심 내용:
- **Power-Interest Matrix**: 이해관계자의 영향력과 관심도에 따른 분류
- **Network Analysis**: 이해관계자 간의 관계와 영향 경로를 분석
- **Centrality Metrics**: Degree, Betweenness, Closeness, Eigenvector
- **Structural Holes**: 분리된 영역 간의 격차를 관리하는 정보 이점
- **Weak Ties**: 새로운 정보와 기회를 가져오는 약한 연결

## Part 3: 기획에서의 활용과 변화 관리

### 정보 흐름 분석 (Information Flow Analysis)

네트워크 분석의 가장 실용적인 활용:
- **여론 주도자(Opinion Leaders) 식별**
  - Degree + Closeness가 높은 사람 = 정보 전파의 중심
  - 이들을 먼저 설득하면 네트워크 전체로 영향 확산

- **정보 경로(Information Pathways) 추적**
  - 특정 정보가 어떤 경로로 전파되는가?
  - Betweenness 높은 노드가 게이트키퍼 역할

- **루머 경로(Rumor Pathways) 차단**
  - 부정적 정보가 어떻게 확산되는가?
  - Betweenness 높은 브로커와의 신뢰 구축으로 루머 차단

### 저항과 지지 분석 (Resistance and Support Analysis)

#### 전략적 연합(Strategic Coalition) 형성

| **전략** | **대상** | **근거** | **목표** |
|---------|---------|---------|----------|
| **조기 도입자 활용** | 지지 + 높은 degree | 영향력 + 신뢰성 | 정책 챔피언 역할 |
| **브로커 설득** | 높은 betweenness | 다층 네트워크 접근 | 진영 간 대화 채널 |
| **커뮤니티별 맞춤** | 같은 커뮤니티 다수 | 집단 효과 | 파벌별 메시징 |
| **반대 진영 약화** | Betweenness 높은 반대자 | 브로커 중보 제거 | 연합 와해 |
| **중간파 포섭** | 중립 + 높은 degree | 유연성 + 영향력 | 캐스팅보트 |

### Rogers(1962)의 혁신 확산 이론

**5가지 채택자 유형** (시간 순서):

1. **Innovators (혁신가, 2.5%)**
   - 새로운 아이디어 좋아함, 위험감수
   - 네트워크 분석: 고도의 연결, 이질적 네트워크

2. **Early Adopters (조기 도입자, 13.5%)**
   - 의견 지도자, 신뢰도 높음
   - 네트워크 분석: **높은 중심성 + 지지 태도** ← 기획의 핵심 대상

3. **Early Majority (초기 다수, 34%)**
   - 신중한 채택, 조기 도입자 신뢰
   - 네트워크 분석: 조기 도입자와의 약한 연결

4. **Late Majority (후기 다수, 34%)**
   - 압박에 의해 채택, 회의적
   - 네트워크 분석: 초기 다수와의 연결

5. **Laggards (낙오자, 16%)**
   - 전통 선호, 마지막에 채택 또는 거부
   - 네트워크 분석: 고립된 네트워크, 반대 진영

### 계단식 효과(Cascade Effect)와 임계점(Tipping Point)

**개념:**
- 초기에 작은 변화가 시간이 지나면서 전체 시스템을 급격히 변화시킴
- 예: 초기 지지자 20% → 50% → 갑자기 80% (S자 곡선)

**네트워크에서의 임계점:**
- 네트워크 밀도가 높을수록 빠른 전파
- Degree 높은 사람들의 연결 가능성이 높을수록 급격한 변화
- 커뮤니티 간 브로커의 역할이 중요

**기획 전략:**
1. 임계점에 도달하기 전에 충분한 지지 기반 구축
2. Betweenness 높은 브로커를 통해 다른 커뮤니티로 확산
3. 초기 도입자들의 success story를 강조 (사회적 증거)

### 변화 관리 전략 매트릭스

| **전략** | **네트워크 분석** | **핵심 활동** | **성공 지표** |
|---------|-----------------|------------|---------------|
| **조기 도입자 활용** | 지지 + 높은 degree + eigenvector | 정책 챔피언 임명, 공개 지지 | 확산 속도 ↑ |
| **브로커 설득** | 높은 betweenness | 1:1 대화, 경청, 우려 반영 | 진영 간 신뢰 ↑ |
| **커뮤니티별 전략** | 로벵 알고리즘 결과 | 맞춤형 메시징, 지역 설명회 | 각 진영 지지도 ↑ |
| **저항자 관리** | 반대 + 높은 power | 우려 해결, 보상 제시 | 반대 전환 또는 중립화 |
| **정보 흐름 관리** | 고도 연결 노드 중심 | 공식 채널 → 비공식 네트워크 확산 | 정보 일관성 ↑ |

In [ ]:
# 변화 관리 시뮬레이션: 초기 도입자부터 시작하여 지지도 확산
# 시간 단계별 지지도 변화 모의

# 초기 상태: 초기 도입자(높은 centrality + 지지)
initial_adopters = centrality_df[(centrality_df['Attitude'] == 'Support') & 
                                   (centrality_df['Eigenvector'] >= centrality_df['Eigenvector'].quantile(0.6))]['ID'].tolist()

print(f"✅ 초기 도입자: {[G.nodes[n]['name_kr'] for n in initial_adopters]}")

# 시간 단계별 확산
adoption = {node: (1 if G.nodes[node]['attitude'] == 1 else -1 if G.nodes[node]['attitude'] == -1 else 0) 
            for node in G.nodes()}

# 시뮬레이션: 5 time steps
time_steps = 5
adoption_history = {t: [] for t in range(time_steps)}

# 초기 상태 기록
for node in G.nodes():
    if G.nodes[node]['attitude'] == 1:
        adoption_history[0].append(1)
    elif G.nodes[node]['attitude'] == -1:
        adoption_history[0].append(-1)
    else:
        adoption_history[0].append(0)

# 실제 확산 시뮬레이션
adoption_state = adoption.copy()
for t in range(1, time_steps):
    new_adoption = adoption_state.copy()
    for node in G.nodes():
        if adoption_state[node] == 0:  # 중립인 경우에만
            # 이웃 중 지지자의 비율 계산
            neighbors = list(G.neighbors(node))
            if neighbors:
                support_count = sum(1 for n in neighbors if adoption_state[n] == 1)
                opposition_count = sum(1 for n in neighbors if adoption_state[n] == -1)
                
                # 임계값: 50% 이상이 지지하면 채택
                if support_count > len(neighbors) * 0.4:
                    new_adoption[node] = 1  # 채택
                elif opposition_count > len(neighbors) * 0.6:
                    new_adoption[node] = -1  # 반대
    
    adoption_state = new_adoption
    for node in G.nodes():
        adoption_history[t].append(adoption_state[node])

# 결과 집계
support_timeline = []
neutral_timeline = []
oppose_timeline = []

for t in range(time_steps):
    support_count = adoption_history[t].count(1)
    neutral_count = adoption_history[t].count(0)
    oppose_count = adoption_history[t].count(-1)
    support_timeline.append(support_count)
    neutral_timeline.append(neutral_count)
    oppose_timeline.append(oppose_count)

# 시각화
fig = go.Figure()

fig.add_trace(go.Scatter(x=list(range(time_steps)), y=support_timeline,
                         mode='lines+markers', name='Support', 
                         line=dict(color='rgba(46, 204, 113, 0.8)', width=3),
                         marker=dict(size=10)))
fig.add_trace(go.Scatter(x=list(range(time_steps)), y=neutral_timeline,
                         mode='lines+markers', name='Neutral',
                         line=dict(color='rgba(149, 165, 166, 0.6)', width=3),
                         marker=dict(size=10)))
fig.add_trace(go.Scatter(x=list(range(time_steps)), y=oppose_timeline,
                         mode='lines+markers', name='Oppose',
                         line=dict(color='rgba(231, 76, 60, 0.8)', width=3),
                         marker=dict(size=10)))

fig.update_layout(
    title='Change Management Simulation: Support Diffusion Over Time',
    xaxis_title='Time Steps',
    yaxis_title='Number of Stakeholders',
    width=900, height=500,
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

print(f"\n📈 시뮬레이션 결과:")
print(f"  • 초기: 지지 {support_timeline[0]}명 ({support_timeline[0]/15*100:.1f}%)")
print(f"  • 최종: 지지 {support_timeline[-1]}명 ({support_timeline[-1]/15*100:.1f}%)")
print(f"  • 변화: {support_timeline[-1] - support_timeline[0]}명 증가")
print(f"\n💡 해석: 초기 도입자 {len(initial_adopters)}명으로부터 시작한 지지가")
print(f"   네트워크를 통해 시간이 지나면서 확산되는 모습을 보여줍니다.")

### 📝 이론 과제 7-2-3 (15분)

#### 과제: 네트워크 기반 변화 관리

**질문:**

1. **Rogers(1962)의 혁신 확산 이론**
   - 5가지 채택자 유형을 나열하고, 각각의 특징을 1-2문장으로 설명하시오.
   - 기획자 입장에서 "Early Adopters"에 집중해야 하는 이유는 무엇인가?

2. **네트워크 기반 변화 관리에서의 역할 비교**
   - "조기 도입자(Early Adopter)"와 "브로커(Broker)"의 역할과 선발 기준의 차이점을 설명하시오.
   - 기획의 초기 단계에서 어떤 사람을 먼저 설득해야 하는가? (2-3 문장)

3. **실제 정책/조직 변화 사례 조사**
   - 당신이 알고 있는 성공한 정책 또는 조직 변화 사례를 찾아, 네트워크 관점에서 분석하시오.
   - 예: 누가 초기 도입자였는가? 누가 브로커였는가? 어떤 커뮤니티가 먼저 채택했는가?

**조사 키워드:**
- "Rogers diffusion of innovations five adopters"
- "network-based change management"
- "successful policy adoption case study"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 7-2-3 제출란

**학번:** ___________
**이름:** ___________

#### 1. Rogers의 혁신 확산 이론과 Early Adopters의 중요성

(여기에 작성)

---

#### 2. 조기 도입자와 브로커의 역할 비교

(여기에 작성)

---

#### 3. 성공한 정책/조직 변화 사례 (네트워크 분석)

(여기에 작성)

## Part 4: 실습 1 - 이해관계자 매핑

### 실습 목표
당신의 선택 프로젝트에 대해:
1. 이해관계자 데이터 수집 (최소 8명)
2. Power-Interest Matrix 작성
3. 태도 분석 및 고위험 이해관계자 식별
4. 전략적 대응 방안 도출

### 프로젝트 선택 제안
- **조직 사례**: 신사업 진출, 업무 프로세스 개선, 기술 도입
- **정책 사례**: 규제 완화, 신규 정책 도입, 지역 개발 사업
- **개인 사례**: 직업 전환, 대학원 진학, 창업 계획

### 실습 단계
1. 이해관계자 리스트 업 (이름, 소속, 역할)
2. Power 평가 (0-100: 권한, 자원 통제, 전문성, 네트워크 종합)
3. Interest 평가 (0-100: 영향 정도, 관심도)
4. Attitude 결정 (1=지지, 0=중립, -1=반대)
5. 4분면 분류 및 전략 수립

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# TODO 1: 당신의 프로젝트에 대한 이해관계자 데이터 생성 (최소 8명)
# 힌트: pd.DataFrame으로 columns는 'name', 'category', 'power', 'interest', 'attitude'
# Power와 Interest는 0-100 범위, attitude는 1(지지), 0(중립), -1(반대)

my_stakeholders = pd.DataFrame({
    'name': [],  # 이해관계자 이름
    'name_kr': [],  # 한글 이름
    'category': [],  # 카테고리 (조직, 정부, 경쟁사 등)
    'power': [],  # 권력 수준 (0-100)
    'interest': [],  # 관심도 (0-100)
    'attitude': []  # 태도 (1: 지지, 0: 중립, -1: 반대)
})

# TODO 2: 4분면 분류 함수 작성
# 힌트: power >= 50 and interest >= 50 → 'Key Players' 등
def classify_quadrant(power, interest):
    """
    Returns quadrant classification based on power and interest
    """
    pass

# my_stakeholders에 quadrant 칼럼 추가
# my_stakeholders['quadrant'] = my_stakeholders.apply(...)

# TODO 3: Power-Interest Matrix 시각화 (Plotly scatter)
# 힌트: 색상은 attitude, 심볼은 category로 구분
# 4분면 구분선을 x=50, y=50에 추가

# fig = go.Figure()
# for idx, row in my_stakeholders.iterrows():
#     fig.add_trace(...)

# TODO 4: 고위험 이해관계자 식별 (높은 권력 + 반대 태도)
# 힌트: my_stakeholders[(power >= 70) & (attitude == -1)]

# TODO 5: 각 4분면별 주요 이해관계자와 전략 정리
# 힌트: groupby('quadrant')로 그룹화

# ========== 여기까지 ==========

print("실습 과제 7-2-4 완료!")

## Part 5: 실습 2 - 네트워크 분석

### 실습 목표
Part 4에서 만든 이해관계자에 대해:
1. 이해관계자 간 관계(edge) 데이터 생성
2. NetworkX로 그래프 구축
3. 중심성 지표 계산
4. 커뮤니티 탐지
5. 네트워크 시각화

### 관계 데이터 수집 기준
- **직접 관계**: 직무 상 정기적 접촉 (가중치 4-5)
- **간접 관계**: 가끔 만나거나 공식 채널 (가중치 2-3)
- **영향 관계**: 한쪽이 다른 쪽에 영향 미침 (방향성 고려)

### 실습 단계
1. 최소 15개의 엣지(관계) 데이터 생성
2. NetworkX Graph 구축
3. 4가지 중심성 지표 계산
4. 상위 5인의 중심성 프로필 비교
5. 커뮤니티 탐지 및 기획 시사점 도출

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# TODO 1: 이해관계자 간 관계 데이터 생성 (최소 15개 엣지)
# 힌트: edges = [('A', 'B', weight), ...]
# 각 엣지는 (source, target, weight) 튜플로 구성
my_edges = [
    # 예시:
    # ('Stakeholder1', 'Stakeholder2', 4),  # 강한 연결
    # ('Stakeholder1', 'Stakeholder3', 2),  # 약한 연결
]

# TODO 2: NetworkX 그래프 구축
my_G = nx.Graph()
# 노드 추가: my_stakeholders의 'name' 칼럼 사용
# for idx, row in my_stakeholders.iterrows():
#     my_G.add_node(...)

# 엣지 추가
# for source, target, weight in my_edges:
#     my_G.add_edge(...)

# TODO 3: 중심성 지표 계산
# my_degree_cent = nx.degree_centrality(my_G)
# my_betweenness_cent = nx.betweenness_centrality(my_G, weight='weight')
# my_closeness_cent = nx.closeness_centrality(my_G, distance='weight')
# my_eigenvector_cent = nx.eigenvector_centrality(my_G, weight='weight', max_iter=500)

# 결과를 DataFrame으로
# my_centrality_df = pd.DataFrame(...)

# TODO 4: 커뮤니티 탐지 (Louvain)
# my_communities = list(louvain_communities(my_G, seed=42))
# print(f"발견된 커뮤니티: {len(my_communities)}개")

# TODO 5: 네트워크 시각화 (Plotly)
# 힌트: spring_layout으로 좌표 계산, go.Scatter로 엣지와 노드 그리기
# pos = nx.spring_layout(my_G, k=2, iterations=50, seed=42)

# TODO 6: 전략적 시사점 도출
# - 핵심 브로커는 누구인가? (betweenness 높은 사람)
# - 조기 도입자로 적합한 사람은? (지지 + 높은 degree)
# - 커뮤니티별 맞춤 전략은 무엇인가?

# ========== 여기까지 ==========

print("실습 과제 7-2-5 완료!")

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# TODO 1: 조기 도입자 식별
# 조건: 지지 태도 + 높은 중심성 (degree, eigenvector)
# 힌트:
# early_adopters = my_centrality_df[
#     (my_stakeholders['attitude'] == 1) & 
#     (my_centrality_df['Degree'] >= ...) & 
#     (my_centrality_df['Eigenvector'] >= ...)
# ].nlargest(3, 'Eigenvector')

# TODO 2: 변화 관리 전략 매트릭스 작성
# 각 전략별로:
# - 대상 이해관계자 (구체적 이름)
# - 선택 근거 (중심성 수치)
# - 권장 활동 (구체적 액션)

strategy_matrix = pd.DataFrame({
    'Strategy': ['Early Adopter Utilization', 'Broker Engagement', 'Community-Specific Approach', 
                 'Resistance Management', 'Information Flow'],
    'Target': [],  # 구체적 이해관계자 이름
    'Evidence': [],  # 근거 (중심성 수치)
    'Recommended Activity': []  # 권장 활동
})

# TODO 3: 전략 매트릭스를 Plotly 테이블로 시각화
# fig = go.Figure(data=[go.Table(...)])

# ========== 여기까지 ==========

print("실습 과제 7-2-6 완료!")

In [ ]:
# ========== 여기서부터 코드를 작성하세요 (보너스) ==========

# TODO: 이해관계자별 참여 전략 히트맵 생성
# X축: 5가지 참여 전략 유형
#   - 정보 제공 (Information Sharing)
#   - 협의 (Consultation)
#   - 참여 (Participation)
#   - 협력 (Collaboration)
#   - 위임 (Delegation)
# Y축: 이해관계자 목록
# 값: 전략 적절성 점수 (1-5, 5=매우 적절)

# 전략 선택 기준:
# - Information Sharing: 낮은 관심도, 낮은 권력
# - Consultation: 높은 관심도, 낮은 권력
# - Participation: 중간 권력, 높은 관심도
# - Collaboration: 높은 권력, 높은 관심도 (지지자)
# - Delegation: 높은 권력, 높은 관심도 (신뢰도 높은 사람)

# 힌트: go.Heatmap을 사용
# strategies = ['Information', 'Consultation', 'Participation', 'Collaboration', 'Delegation']
# heatmap_data = [[...] for each stakeholder]

# fig = go.Figure(data=go.Heatmap(
#     z=heatmap_data,
#     x=strategies,
#     y=my_stakeholders['name_kr'],
#     colorscale='RdYlGn'
# ))

# ========== 여기까지 ==========

print("실습 과제 7-2-7 (보너스) 완료!")

## 🎓 강의 마무리 및 핵심 요약

### 오늘 배운 내용

1. **이해관계자 분석의 기초**
   - Freeman의 정의: 조직의 목표 달성에 영향을 미치거나 영향을 받는 개인/집단
   - 전통적 주주중심 경영의 한계를 극복하는 관점

2. **Power-Interest Matrix의 활용과 한계**
   - 4분면 분류로 초기 이해관계자 파악 가능
   - 하지만 태도, 관계, 변화 가능성을 반영하지 못함
   - **네트워크 분석의 필요성** ← 오늘의 핵심

3. **사회 연결망 분석(SNA)의 핵심 개념**
   - **Degree**: 얼마나 많은 사람과 연결되어 있는가? → 영향 범위
   - **Betweenness**: 다른 사람 간의 중개자인가? → 정보 통제력, 갈등 중보
   - **Closeness**: 네트워크 전체와 얼마나 가까운가? → 리더십 잠재력
   - **Eigenvector**: 중요 노드와 연결되어 있는가? → 권력 기반

4. **Structural Holes와 Weak Ties의 전략적 활용**
   - Burt: 분리된 영역을 연결하는 사람이 정보, 통제, 시각의 이점을 가짐
   - Granovetter: 약한 연결이 새로운 정보와 기회를 가져옴
   - 기획자는 브로커 역할 + 약한 연결 파트너 적극 활용

5. **네트워크 기반 변화 관리**
   - Rogers의 5가지 채택자 유형: Innovators → Early Adopters → ... → Laggards
   - **Early Adopters 중심 전략**: 신뢰도 높은 의견 지도자를 먼저 설득
   - **Broker 설득**: 여러 진영을 연결하는 사람이 정책 전파의 키
   - **Community Detection**: 파벌/연합 자동 식별 → 맞춤형 전략
   - **Cascade Effect**: 임계점(Tipping Point)에 도달하면 급격한 변화

### 핵심 메시지

> **기획은 혼자 하는 것이 아니다.**
>
> 아무리 완벽한 계획이라도 사람들의 지지 없이는 실행되지 않는다.
>
> 이해관계자를 단순히 분류하지 말고, **그들 간의 관계와 네트워크를 이해하라.**
>
> 영향력은 직위가 아니라 **관계 속에서 발생**한다.
>
> 초기 도입자 3-5명을 설득하고 그들의 네트워크를 활용하면,
>
> 조직 전체의 변화를 만들 수 있다.

### 다음 장 예고

**8장: 베이지안 실험설계와 정책 평가**
- 이해관계자의 불확실성을 확률로 다루기
- A/B 테스트와 인과추론
- 정책 효과 측정
- 이해관계자 네트워크 구조가 시나리오별로 어떻게 변화하는지 추적

### 📚 추가 학습 자료

**참고문헌:**
- Freeman, R.E. (1984). *Strategic Management: A Stakeholder Approach*. Pitman Publishing.
- Burt, R.S. (1992). *Structural Holes: The Social Structure of Competition*. Harvard University Press.
- Granovetter, M.S. (1973). "The Strength of Weak Ties". *American Journal of Sociology*, 78(6), 1360-1380.
- Mendelow, A.L. (1991). "Environmental Scanning: The Impact of the CEO's Perceptions". In ICIS 1991 Proceedings.
- Rogers, E.M. (1962). *Diffusion of Innovations* (5th ed., 2003). Free Press.
- Centola, D. (2010). "The Spread of Behavior in an Online Social Network Experiment". *Science*, 329(5996), 1194-1197.

**Python 라이브러리:**
- NetworkX: https://networkx.org/ (그래프/네트워크 분석)
- Plotly: https://plotly.com/python/ (대화형 시각화)
- Pandas: https://pandas.pydata.org/ (데이터 처리)

**온라인 리소스:**
- NetworkX 튜토리얼: https://networkx.org/documentation/stable/tutorial.html
- "Introduction to Social Network Analysis" (SNA 개론)
- "Network Science" by Albert-László Barabási (고급 네트워크 과학)

### 자기평가 체크리스트

강의를 마친 후 다음을 확인해 보세요:

- [ ] Freeman의 이해관계자 정의를 설명할 수 있는가?
- [ ] Power-Interest Matrix의 4분면을 구분할 수 있는가?
- [ ] Degree, Betweenness, Closeness, Eigenvector의 차이를 설명할 수 있는가?
- [ ] Structural Holes와 Weak Ties를 구분할 수 있는가?
- [ ] Rogers의 채택자 유형 5가지를 나열할 수 있는가?
- [ ] NetworkX로 간단한 네트워크를 구축하고 중심성을 계산할 수 있는가?
- [ ] 실제 프로젝트의 이해관계자 네트워크를 분석할 수 있는가?
- [ ] 네트워크 분석 결과를 기획 전략으로 전환할 수 있는가?